In [ ]:
import sys
sys.path.insert(0, '/var/www/python/Qingcheng/Darwin')
sys.path.append('/var/www/python/Prod/nighthawk/')
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from nighthawk.util import bigquery_functions, connections, sql_functions
from google.cloud import bigquery
import utils_darwin.node_price_predictor_darwin as node_price_predictor_darwin


# --- config ---
start_date    = '2026-01-10'
end_date      = '2026-03-30'
run_number    = 1
opexchange    = 'SPP'

if int(run_number) == 1:
    data_location = 'training'
    load_model    = False
else:
    data_location = 'secondRun'
    load_model    = True

work_str = '''
import pandas as pd
import numpy as np
import time
import sys
import pickle
import dill
import warnings
from google.cloud import bigquery, storage
warnings.filterwarnings("ignore")
from datetime import datetime
print("Here the job starts!")
print(str(datetime.now()))

bucket_name      = sys.argv[1]
save_file_folder = sys.argv[2]
job_table_id     = sys.argv[3]
with open("job_info.pickle", "rb") as file:
    job_info = dill.load(file)

job_id    = job_info["jobId"]
record_id = job_info["recordId"]
node_num  = job_info["node_num"]
dt        = job_info['dt']
# **********************do not change code above**********************************

import sys
try:
    sys.path.append("/var/www/python/Prod/nighthawk/")
except:
    print("oh..this is in kubernetes")

import ve_model_functions_darwin

opexchange     = "SPP"
location       = "gs://ve_fourier/production/SPP/''' + data_location + '''"
model_location = "gs://ve_darwin/production/" + opexchange + "/models"
data_df = pd.read_csv(location + "/{node_num}_{bid_date}.csv".format(node_num=node_num, bid_date=dt))

south_broadzonelevel_hdd_threshold = 42
north_broadzonelevel_hdd_threshold = 70
exclude_feb2021_data_in_train = True
if "spp_south_broadzonelevel_hdd_forecast_f" in data_df.columns:
    if np.unique(data_df.loc[data_df["dt"] == dt, "spp_south_broadzonelevel_hdd_forecast_f"])[0] >= south_broadzonelevel_hdd_threshold:
        exclude_feb2021_data_in_train = False
elif "spp_north_broadzonelevel_hdd_forecast_f" in data_df.columns:
    if np.unique(data_df.loc[data_df["dt"] == dt, "spp_north_broadzonelevel_hdd_forecast_f"])[0] >= north_broadzonelevel_hdd_threshold:
        exclude_feb2021_data_in_train = False
feb2021_dates = pd.DataFrame(pd.date_range("2021-02-13", "2021-02-19", freq="D"), columns=["dt"])["dt"].dt.strftime("%Y-%m-%d").tolist()
if exclude_feb2021_data_in_train and (dt not in feb2021_dates):
    data_df = data_df[~data_df["dt"].isin(feb2021_dates)]

res_df              = pd.DataFrame()
imp_df              = pd.DataFrame()
additional_preds_df = pd.DataFrame()
for y_var in [["rt_congestion"], ["da_congestion"], ["rt_total"], ["da_total"]]:
    for var_list in [["All"]]:
        trainingPeriod = int(min((pd.to_datetime(dt) - pd.to_datetime("2017-01-01")) / np.timedelta64(1, "D"), 730))
        xtrain, ytrain, xtest, ytest = ve_model_functions_darwin.getTrainTestData(
            opexchange, data_df, y=y_var, bidDt=dt,
            trainingPeriod=str(trainingPeriod) + "D", train_a_or_f="f")

        rf_result_dict = ve_model_functions_darwin.buildRandomForestModel(
            opexchange, xtrain, ytrain, xtest, ytest, node_num, dt,
            reweight_period="15D", reweight_equivalent="600D", calc_permutation_imp=False, verbose=0, n_estimators=500,
            n_jobs=-1, max_features="sqrt", bootstrap=True, oob_score=True, oob_scr_recent_days="30D",
            criterion="mse", max_depth=None, min_samples_split=15,
            quantiles=[0.01, 0.03, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.97, 0.99],
            calc_leaveout_preds=True, calc_last_nday_insample_preds=True, nday_insample_preds=[3],
            save_model=True, load_model=''' + str(load_model) + ''',
            gcs_model_location=model_location + "/rfmodel_{node_num}_{bid_date}_{model}.pkl".format(
                node_num=node_num, bid_date=dt, model="_".join(y_var)))

        res_, imp_, mdl, perm_imp_, additional_preds_ = (
            rf_result_dict["predictions"], rf_result_dict["feature_importance"],
            rf_result_dict["trained_model"], rf_result_dict["permutation_importance"],
            rf_result_dict["additional_preds"])
        res_["keep_list"] = "_".join(var_list)
        res_["y_list"]    = "_".join(y_var)
        res_df = pd.concat([res_df, res_])
        imp_["keep_list"] = "_".join(var_list)
        imp_["y_list"]    = "_".join(y_var)
        imp_df = pd.concat([imp_df, imp_])
        additional_preds_["keep_list"] = "_".join(var_list)
        additional_preds_["y_list"]    = "_".join(y_var)
        additional_preds_df = pd.concat([additional_preds_df, additional_preds_])

df  = res_df
df2 = imp_df
df3 = additional_preds_df

basic_price_cols = ["da_total", "rt_total", "da_congestion", "rt_congestion", "da_slack", "rt_slack"]
ll  = [x for x in df.columns  if x not in basic_price_cols]
df  = pd.merge(df[ll],  data_df[["node_num", "dt", "hr"] + basic_price_cols].drop_duplicates().dropna(), how="left", on=["node_num", "dt", "hr"])
ll3 = [x for x in df3.columns if x not in basic_price_cols]
df3 = pd.merge(df3[ll3], data_df[["node_num", "dt", "hr"] + basic_price_cols].drop_duplicates().dropna(), how="left", on=["node_num", "dt", "hr"])

# **********************do not change code below**********************************
testing_data_file_name = "gs://" + bucket_name + "/" + save_file_folder + "/record_" + str(record_id) + "/prediction/jobId_" + str(job_id) + ".csv"
print(testing_data_file_name)
df.to_csv(testing_data_file_name, index=False)

testing_data_file_name = "gs://" + bucket_name + "/" + save_file_folder + "/record_" + str(record_id) + "/meta/jobId_" + str(job_id) + ".csv"
print(testing_data_file_name)
df2.to_csv(testing_data_file_name, index=False)

testing_data_file_name = "gs://" + bucket_name + "/" + save_file_folder + "/record_" + str(record_id) + "/additionalprediction/jobId_" + str(job_id) + ".csv"
print(testing_data_file_name)
df3.to_csv(testing_data_file_name, index=False)

print("Here the job ends!")
print(str(datetime.now()))
'''

# --- pull node list from Darwin ---
conn = connections.get_sql_connection(database='Darwin_' + opexchange.upper())
daily_node_df = pd.read_sql(
    f"SELECT DISTINCT dt, node_num FROM Darwin_{opexchange}.nodeSelection WHERE dt >= '{start_date}' AND dt <= '{end_date}'", conn)
daily_node_df['dt'] = daily_node_df['dt'].astype(str)
conn.dispose()
if daily_node_df.empty:
    print('  No nodes found, skipping.')
print(f'  {len(daily_node_df)} node-date pairs')

npp = node_price_predictor_darwin.NodePricePredictorDarwin(opexchange, daily_node_df[['dt', 'node_num']])
record_id = npp.get_record_id()
print('  record_id', record_id)

yaml_dict = dict()
yaml_dict["spec"] = dict()
yaml_dict["spec"]["parallelism"] = 80
yaml_dict["spec"]["template"] = dict()
yaml_dict["spec"]["template"]["spec"] = dict()
yaml_dict["spec"]["template"]["spec"]["containers"] = [dict()]
yaml_dict["spec"]["template"]["spec"]["containers"][0]["resources"] = dict()
yaml_dict["spec"]["template"]["spec"]["containers"][0]["resources"]["limits"]   = {"cpu": "1.5"}
yaml_dict["spec"]["template"]["spec"]["containers"][0]["resources"]["requests"] = {"cpu": "1"}
yaml_dict["spec"]["template"]["spec"]["nodeSelector"] = {"cloud.google.com/gke-nodepool": "ve-pool-e2std4"}

file_location = '/var/www/python/Qingcheng/Darwin/utils_darwin/ve_model_functions_darwin.py'
print(file_location)

predict_table = npp.run(work_str, envir='KUBERNETES', scale='FULL',
                        initialization='us-central1-docker.pkg.dev/movetocloud-999/darwin/darwinbase',
                        yaml_dict_original=yaml_dict,
                        other_folder=file_location,
                        install_basic_pkgs=False)
print('  predict_table:', predict_table)

prediction_table  = predict_table[0]
importance_table  = predict_table[1]
additional_table  = predict_table[2]

# delete existing backtest records then append
try:
    client = bigquery.Client()
    job = client.query(
        f"DELETE FROM `Darwin_Production.{opexchange}_prediction` WHERE dt >= '{start_date}' AND dt <= '{end_date}' AND run_number = {run_number}")
    job.result()
    job = client.query(
        f"DELETE FROM `Darwin_Production.{opexchange}_importance` WHERE dt >= '{start_date}' AND dt <= '{end_date}'")
    job.result()
    job = client.query(
        f"DELETE FROM `Darwin_Production.{opexchange}_lookbackPrediction` WHERE dt >= '{start_date}' AND dt <= '{end_date}'")
    job.result()
except Exception:
    pass

bigquery_functions.create_bq_table_from_query(
    f"SELECT {run_number} AS run_number, * EXCEPT(da_total, rt_total, da_congestion, rt_congestion, da_slack, rt_slack) FROM `{prediction_table}`",
    dataset_name='Darwin_Production', table_name=f'{opexchange}_prediction',
    partition_column='dt', write_disposition='WRITE_APPEND')

bigquery_functions.create_bq_table_from_query(
    f"SELECT * FROM `{importance_table}`",
    dataset_name='Darwin_Production', table_name=f'{opexchange}_importance',
    partition_column='dt', write_disposition='WRITE_APPEND')

bigquery_functions.create_bq_table_from_query(
    f"SELECT {run_number} AS run_number, * FROM `{additional_table}` WHERE pred_type = 'insample_last_nday_3'",
    dataset_name='Darwin_Production', table_name=f'{opexchange}_lookbackPrediction',
    partition_column='dt', write_disposition='WRITE_APPEND')

# save local CSV
df = bigquery_functions.download_df_from_bq(f"SELECT * FROM `{prediction_table}`")
out_path = f'/var/www/python/Qingcheng/WFiles/spp_darwin_rf_{start_date}_{end_date}.csv'
df.to_csv(out_path, index=False)
print(f'Saved to {out_path}: {df.shape}')
